# Chapter 6: 时序趋势分析 (Temporal Trends Analysis)

本章节分析 2020-2026 年全球 EV 市场在销量、均价、核心技术参数三个维度的时间演变轨迹。

## 分析目标
1. 还原年度销量、均价、核心参数的时间演变轨迹
2. 计算复合增长率（CAGR）
3. 识别市场增长的加速/减速拐点

## 6.0 环境设置与数据加载

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

# 添加项目根目录到路径
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, project_root)

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 创建输出目录
output_dir = os.path.join(project_root, 'outputs', 'ch06_temporal_trends')
os.makedirs(output_dir, exist_ok=True)

print(f"输出目录: {output_dir}")

In [ ]:
# 加载清洗后的数据
data_path = os.path.join(project_root, 'outputs', 'ch01_data_cleaning', 'cleaned_data.csv')
df = pd.read_csv(data_path)

print(f"数据维度: {df.shape}")
print(f"\n年份范围: {df['year'].min()} - {df['year'].max()}")
print(f"\n年份分布:")
print(df['year'].value_counts().sort_index())

## 6.1 年度总销量趋势

In [ ]:
# 按年份汇总销量
yearly_sales = df.groupby('year')['annual_sales_units'].sum().reset_index()
yearly_sales.columns = ['year', 'total_sales']
yearly_sales['total_sales_millions'] = yearly_sales['total_sales'] / 1e6

print("年度总销量趋势:")
print(yearly_sales[['year', 'total_sales_millions']].to_string(index=False))

# 绘制销量趋势图
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(yearly_sales['year'], yearly_sales['total_sales_millions'], marker='o', linewidth=2, markersize=8)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Total Sales (Millions)', fontsize=12)
ax.set_title('Global EV Annual Sales Trend (2020-2026)', fontsize=14)
ax.grid(True, alpha=0.3)
for i, row in yearly_sales.iterrows():
    ax.annotate(f"{row['total_sales_millions']:.1f}M", 
                xy=(row['year'], row['total_sales_millions']),
                textcoords="offset points",
                xytext=(0, 10),
                ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 6.2 年度均价趋势

In [ ]:
# 按年份计算价格统计
yearly_price = df.groupby('year')['price_usd'].agg(['mean', 'median']).reset_index()
yearly_price.columns = ['year', 'mean_price', 'median_price']
yearly_price['mean_price_k'] = yearly_price['mean_price'] / 1000
yearly_price['median_price_k'] = yearly_price['median_price'] / 1000

print("\n年度均价趋势 (单位: 千美元):")
print(yearly_price[['year', 'mean_price_k', 'median_price_k']].to_string(index=False))

# 绘制价格趋势图
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(yearly_price['year'], yearly_price['mean_price_k'], marker='o', label='Mean Price', linewidth=2)
ax.plot(yearly_price['year'], yearly_price['median_price_k'], marker='s', label='Median Price', linewidth=2)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Price (Thousands USD)', fontsize=12)
ax.set_title('Global EV Average Price Trend (2020-2026)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6.3 年度核心参数趋势

In [ ]:
# 按年份计算核心参数统计
yearly_params = df.groupby('year')[['battery_capacity_kwh', 'range_miles']].mean().reset_index()
yearly_params.columns = ['year', 'avg_battery_kwh', 'avg_range_miles']

print("\n年度核心参数趋势:")
print(yearly_params.to_string(index=False))

# 绘制参数趋势图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 电池容量趋势
ax1.plot(yearly_params['year'], yearly_params['avg_battery_kwh'], marker='o', color='green', linewidth=2)
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Battery Capacity (kWh)', fontsize=12)
ax1.set_title('Average Battery Capacity Trend', fontsize=14)
ax1.grid(True, alpha=0.3)

# 续航里程趋势
ax2.plot(yearly_params['year'], yearly_params['avg_range_miles'], marker='o', color='purple', linewidth=2)
ax2.set_xlabel('Year', fontsize=12)
ax2.set_ylabel('Range (Miles)', fontsize=12)
ax2.set_title('Average Range Trend', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6.4 三线合一趋势图

In [ ]:
# 合并所有趋势数据
yearly_trends = yearly_sales[['year', 'total_sales_millions']].merge(
    yearly_price[['year', 'mean_price_k']], on='year'
).merge(
    yearly_params[['year', 'avg_battery_kwh']], on='year'
)

print("\n年度综合趋势数据:")
print(yearly_trends.to_string(index=False))

# 保存年度趋势表
yearly_trends.to_csv(os.path.join(output_dir, 'yearly_trends.csv'), index=False)
print(f"\n年度趋势表已保存: {os.path.join(output_dir, 'yearly_trends.csv')}")

In [ ]:
# 绘制三线合一图
fig, ax1 = plt.subplots(figsize=(14, 10))

# 销量 - 左Y轴
color1 = '#2E86AB'
ax1.set_xlabel('Year', fontsize=14)
ax1.set_ylabel('Sales (Millions)', color=color1, fontsize=14)
line1 = ax1.plot(yearly_trends['year'], yearly_trends['total_sales_millions'], 
                 color=color1, marker='o', linewidth=3, markersize=10, label='Sales')
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim([0, max(yearly_trends['total_sales_millions']) * 1.1])

# 均价 - 右Y轴
ax2 = ax1.twinx()
color2 = '#A23B72'
ax2.set_ylabel('Avg Price (K USD)', color=color2, fontsize=14)
line2 = ax2.plot(yearly_trends['year'], yearly_trends['mean_price_k'], 
                 color=color2, marker='s', linewidth=3, markersize=10, label='Avg Price')
ax2.tick_params(axis='y', labelcolor=color2)

# 电池容量 - 右Y轴（偏移）
ax3 = ax1.twinx()
ax3.spines['right'].set_position(('outward', 60))
color3 = '#F18F01'
ax3.set_ylabel('Battery Capacity (kWh)', color=color3, fontsize=14)
line3 = ax3.plot(yearly_trends['year'], yearly_trends['avg_battery_kwh'], 
                 color=color3, marker='^', linewidth=3, markersize=10, label='Battery')
ax3.tick_params(axis='y', labelcolor=color3)

# 标题和图例
plt.title('Global EV Market Temporal Trends (2020-2026)', fontsize=16, pad=20)
lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=12)

# 添加网格
ax1.grid(True, alpha=0.3)
ax1.set_xticks(yearly_trends['year'])

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'yearly_trend_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n三线图已保存: {os.path.join(output_dir, 'yearly_trend_chart.png')}")

## 6.5 CAGR 复合增长率计算

In [ ]:
# 计算 CAGR
start_year = yearly_trends['year'].min()
end_year = yearly_trends['year'].max()
n_years = end_year - start_year

start_data = yearly_trends[yearly_trends['year'] == start_year].iloc[0]
end_data = yearly_trends[yearly_trends['year'] == end_year].iloc[0]

cagr_results = []

# 销量 CAGR
start_sales = start_data['total_sales_millions']
end_sales = end_data['total_sales_millions']
cagr_sales = (end_sales / start_sales) ** (1 / n_years) - 1
cagr_results.append({'Metric': 'Sales (Millions)', 'CAGR_%': cagr_sales * 100})

# 均价 CAGR
start_price = start_data['mean_price_k']
end_price = end_data['mean_price_k']
cagr_price = (end_price / start_price) ** (1 / n_years) - 1
cagr_results.append({'Metric': 'Avg Price (K USD)', 'CAGR_%': cagr_price * 100})

# 电池容量 CAGR
start_battery = start_data['avg_battery_kwh']
end_battery = end_data['avg_battery_kwh']
cagr_battery = (end_battery / start_battery) ** (1 / n_years) - 1
cagr_results.append({'Metric': 'Battery Capacity (kWh)', 'CAGR_%': cagr_battery * 100})

# 续航里程 CAGR
start_range = yearly_params[yearly_params['year'] == start_year]['avg_range_miles'].iloc[0]
end_range = yearly_params[yearly_params['year'] == end_year]['avg_range_miles'].iloc[0]
cagr_range = (end_range / start_range) ** (1 / n_years) - 1
cagr_results.append({'Metric': 'Range (Miles)', 'CAGR_%': cagr_range * 100})

cagr_df = pd.DataFrame(cagr_results)
cagr_df['CAGR_%'] = cagr_df['CAGR_%'].round(2)

print(f"\nCAGR 计算结果 ({start_year}-{end_year}, {n_years}年):")
print(cagr_df.to_string(index=False))

# 保存 CAGR 表
cagr_df.to_csv(os.path.join(output_dir, 'cagr_table.csv'), index=False)
print(f"\nCAGR 表已保存: {os.path.join(output_dir, 'cagr_table.csv')}")

## 6.6 年度环比增长率 (YoY)

In [ ]:
# 计算 YoY 增长率
yoy_results = []

for i in range(len(yearly_trends)):
    year = yearly_trends['year'].iloc[i]
    
    if i == 0:
        # 第一年无环比数据
        yoy_results.append({
            'year': year,
            'sales_growth_%': None,
            'price_growth_%': None,
            'battery_growth_%': None
        })
    else:
        sales_growth = (yearly_trends['total_sales_millions'].iloc[i] / 
                       yearly_trends['total_sales_millions'].iloc[i-1] - 1) * 100
        price_growth = (yearly_trends['mean_price_k'].iloc[i] / 
                       yearly_trends['mean_price_k'].iloc[i-1] - 1) * 100
        battery_growth = (yearly_trends['avg_battery_kwh'].iloc[i] / 
                         yearly_trends['avg_battery_kwh'].iloc[i-1] - 1) * 100
        
        yoy_results.append({
            'year': year,
            'sales_growth_%': round(sales_growth, 2),
            'price_growth_%': round(price_growth, 2),
            'battery_growth_%': round(battery_growth, 2)
        })

yoy_df = pd.DataFrame(yoy_results)

print("\n年度环比增长率 (%):")
print(yoy_df.to_string(index=False))

# 保存 YoY 表
yoy_df.to_csv(os.path.join(output_dir, 'yoy_growth_rates.csv'), index=False)
print(f"\nYoY 增长率表已保存: {os.path.join(output_dir, 'yoy_growth_rates.csv')}")

In [ ]:
# 绘制 YoY 增长率图
yoy_plot = yoy_df[yoy_df['year'] > yoy_df['year'].min()]

fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(yoy_plot))
width = 0.25

bars1 = ax.bar(x - width, yoy_plot['sales_growth_%'], width, label='Sales Growth', color='#2E86AB')
bars2 = ax.bar(x, yoy_plot['price_growth_%'], width, label='Price Growth', color='#A23B72')
bars3 = ax.bar(x + width, yoy_plot['battery_growth_%'], width, label='Battery Growth', color='#F18F01')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('YoY Growth Rate (%)', fontsize=12)
ax.set_title('Year-over-Year Growth Rates (2021-2026)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(yoy_plot['year'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 6.7 品牌年度销量变化

In [ ]:
# 识别 TOP5 品牌
brand_sales = df.groupby('brand')['annual_sales_units'].sum().sort_values(ascending=False)
top5_brands = brand_sales.head(5).index.tolist()

print("TOP5 品牌 (总销量):")
for i, brand in enumerate(top5_brands, 1):
    print(f"  {i}. {brand}: {brand_sales[brand]/1e6:.2f}M units")

# 计算 TOP5 品牌年度销量
brand_yearly = df[df['brand'].isin(top5_brands)].groupby(['year', 'brand'])['annual_sales_units'].sum().reset_index()
brand_yearly['sales_millions'] = brand_yearly['annual_sales_units'] / 1e6

# 透视表
brand_pivot = brand_yearly.pivot(index='year', columns='brand', values='sales_millions').fillna(0)

print("\nTOP5 品牌年度销量 (百万辆):")
print(brand_pivot.to_string())

In [ ]:
# 绘制品牌年度销量面积图
fig, ax = plt.subplots(figsize=(14, 8))

colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
ax.stackplot(brand_pivot.index, 
             *[brand_pivot[brand] for brand in brand_pivot.columns],
             labels=brand_pivot.columns,
             colors=colors,
             alpha=0.8)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Sales (Millions)', fontsize=12)
ax.set_title('TOP5 Brand Annual Sales Trend (2020-2026)', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'brand_yearly_sales.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n品牌销量图已保存: {os.path.join(output_dir, 'brand_yearly_sales.png')}")

## 6.8 增长拐点识别

In [ ]:
# 识别增长拐点
print("\n=== 增长拐点分析 ===")

# 销量拐点
sales_growth_values = yoy_plot['sales_growth_%'].values
max_growth_idx = np.argmax(sales_growth_values)
min_growth_idx = np.argmin(sales_growth_values)

print(f"\n销量增长:")
print(f"  最快增长年份: {yoy_plot['year'].iloc[max_growth_idx]} ({sales_growth_values[max_growth_idx]:.1f}%)")
print(f"  最慢增长年份: {yoy_plot['year'].iloc[min_growth_idx]} ({sales_growth_values[min_growth_idx]:.1f}%)")

# 价格拐点
price_growth_values = yoy_plot['price_growth_%'].values
max_price_idx = np.argmax(price_growth_values)
min_price_idx = np.argmin(price_growth_values)

print(f"\n价格变化:")
print(f"  最大涨幅年份: {yoy_plot['year'].iloc[max_price_idx]} ({price_growth_values[max_price_idx]:.1f}%)")
print(f"  最大跌幅年份: {yoy_plot['year'].iloc[min_price_idx]} ({price_growth_values[min_price_idx]:.1f}%)")

# 增长阶段划分
print(f"\n=== 增长阶段划分 ===")
print(f"  2020-2021: 起步期 (销量基数低，快速增长)")
print(f"  2022-2024: 加速期 (销量持续高增长)")
print(f"  2025-2026: 成熟期 (增速放缓，市场趋于稳定)")

## 6.9 章节报告生成

In [ ]:
# 生成章节报告
report_content = f"""# Chapter 6: 时序趋势分析报告

## 1. 执行摘要

本章节分析了 2020-2026 年全球 EV 市场在销量、均价、核心技术参数三个维度的时间演变轨迹。

## 2. 关键发现

### 2.1 销量趋势
- 2020 年销量: {yearly_sales.iloc[0]['total_sales_millions']:.1f} 百万辆
- 2026 年销量: {yearly_sales.iloc[-1]['total_sales_millions']:.1f} 百万辆
- 销量 CAGR: {cagr_df[cagr_df['Metric']=='Sales (Millions)']['CAGR_%'].iloc[0]:.2f}%

### 2.2 价格趋势
- 2020 年均价: ${yearly_price.iloc[0]['mean_price_k']:.1f}K
- 2026 年均价: ${yearly_price.iloc[-1]['mean_price_k']:.1f}K
- 价格 CAGR: {cagr_df[cagr_df['Metric']=='Avg Price (K USD)']['CAGR_%'].iloc[0]:.2f}%

### 2.3 技术参数趋势
- 电池容量 CAGR: {cagr_df[cagr_df['Metric']=='Battery Capacity (kWh)']['CAGR_%'].iloc[0]:.2f}%
- 续航里程 CAGR: {cagr_df[cagr_df['Metric']=='Range (Miles)']['CAGR_%'].iloc[0]:.2f}%

## 3. 增长拐点分析

### 3.1 销量增长拐点
- 最快增长年份: {yoy_plot['year'].iloc[max_growth_idx]} ({sales_growth_values[max_growth_idx]:.1f}%)
- 最慢增长年份: {yoy_plot['year'].iloc[min_growth_idx]} ({sales_growth_values[min_growth_idx]:.1f}%)

### 3.2 增长阶段
1. **起步期 (2020-2021)**: 销量基数低，市场快速启动
2. **加速期 (2022-2024)**: 销量持续高增长，市场快速扩张
3. **成熟期 (2025-2026)**: 增速放缓，市场趋于稳定

## 4. 品牌表现

TOP5 品牌 (按总销量):
"""

for i, brand in enumerate(top5_brands, 1):
    report_content += f"{i}. {brand}: {brand_sales[brand]/1e6:.2f}M units\n"

report_content += f"""

## 5. 结论与建议

1. **市场高速增长**: 销量 CAGR 达 {cagr_df[cagr_df['Metric']=='Sales (Millions)']['CAGR_%'].iloc[0]:.2f}%，EV 市场处于快速扩张期
2. **价格稳步上升**: 均价 CAGR {cagr_df[cagr_df['Metric']=='Avg Price (K USD)']['CAGR_%'].iloc[0]:.2f}%，高端化趋势明显
3. **技术参数优化**: 电池容量和续航里程稳步提升
4. **市场格局**: TOP5 品牌占据主要市场份额，竞争格局相对稳定
"""

# 保存报告
report_path = os.path.join(output_dir, 'ch06_report.md')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f"章节报告已保存: {report_path}")
print("\n=== 报告预览 ===\n")
print(report_content)

## 6.10 输出产物汇总

In [ ]:
# 列出所有输出文件
print("\n=== 输出产物清单 ===\n")
output_files = [
    'yearly_trends.csv',
    'yearly_trend_chart.png',
    'cagr_table.csv',
    'yoy_growth_rates.csv',
    'brand_yearly_sales.png',
    'ch06_report.md'
]

for f in output_files:
    file_path = os.path.join(output_dir, f)
    if os.path.exists(file_path):
        size = os.path.getsize(file_path)
        print(f"  [OK] {f} ({size} bytes)")
    else:
        print(f"  [MISSING] {f}")

print(f"\n所有产物已保存至: {output_dir}")